# RAG Inference Notebook

Notebook for retrieval-augmented inference on the Austrian tax law dataset.

Workflow:
- load the instruction model in 8-bit mode
- build or load a Chroma vector database
- retrieve matching RIS text chunks
- generate grounded answers
- save answers together with retrieved metadata

## 1. Imports and configuration

This section defines the required libraries, file paths, and core runtime settings.

In [ ]:
import os
import re
import time
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

CSV_PATH = "./data/dataset_clean_worktime_version.csv"
RIS_DOCS_DIR = "./rag_data"
CHROMA_DIR = "./rag_data/chroma_tagged_db"
OUT_PATH = "./outputs/inference_results_rag.csv"

BATCH_SIZE = 4
TOP_K = 3
FORCE_REBUILD_DB = True
MAX_INPUT_LENGTH = 1800
MAX_NEW_TOKENS = 80

## 2. Helper functions

These utilities keep the main inference loop shorter and easier to read.

In [ ]:
def clean_text(text: str) -> str:
    # Normalize whitespace and return a clean one-line string.
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_law_reference(text: str) -> str:
    """Extract a simple legal reference such as a paragraph and law name."""
    text = str(text)

    paragraph_match = re.search(r"(§\s*\d+[a-zA-Z]?)", text, re.IGNORECASE)
    law_match = re.search(
        r"\b((?:KStG|EStG|UStG|GrEStG)(?:\s*19\d\d)?)\b", text, re.IGNORECASE,)

    paragraph = paragraph_match.group(1) if paragraph_match else ""
    law_name = law_match.group(1) if law_match else ""

    result = f"{paragraph} {law_name}".strip()
    return result if result else "nothing_found"


def build_chat_text(tokenizer, system_prompt: str, user_prompt: str) -> str:
    """Apply the model chat template and return plain text."""
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

## 3. Load tokenizer and model

The base model is loaded in 8-bit mode to reduce VRAM usage during inference.

In [ ]:
print("Loading tokenizer and model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available; GPU required for inference")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

quant_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto",)
model.eval()

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print()

## 4. Build or load the vector database

Each RIS text file is split into chunks and stored in Chroma together with:
- source: the original file name
- tag: a short identifier from the document used for filtered retrieval

In [ ]:
print("Loading vector database...")

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, model_kwargs={"device": "cuda"})

if FORCE_REBUILD_DB or not os.path.exists(CHROMA_DIR):
    documents = []
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

    for file_name in os.listdir(RIS_DOCS_DIR):
        if not file_name.endswith(".txt"):
            continue

        file_path = os.path.join(RIS_DOCS_DIR, file_name)
        with open(file_path, "r", encoding="utf-8-sig") as f:
            text = f.read().strip()

        lines = [line.strip() for line in text.splitlines() if line.strip() != ""]
        tag = lines[1] if len(lines) > 1 else ""

        chunks = splitter.split_text(text)

        for chunk in chunks:
            documents.append(
                Document(page_content=chunk, metadata={"source": file_name, "tag": tag}))

    db = Chroma.from_documents(documents, embeddings, persist_directory=CHROMA_DIR)

    print("Vector database rebuilt.")
    print("Documents stored:", len(documents))
else:
    db = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
    print("Existing vector database loaded.")

print()

## 5. Load input prompts

The input CSV is expected to contain:
- ID
- Prompt

The notebook also supports resuming from an existing output file.

In [ ]:
print("Loading input CSV...")

rows = []
with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
    lines = f.readlines()

for line in lines[1:]:
    parts = line.strip().split(",", 1)

    if len(parts) != 2:
        continue

    current_id = parts[0].strip(' "')
    current_prompt = parts[1].strip(' "')

    if current_id and current_prompt:
        rows.append({"ID": current_id, "Prompt": current_prompt})

input_df = pd.DataFrame(rows)

if os.path.exists(OUT_PATH):
    existing_df = pd.read_csv(OUT_PATH, encoding="utf-8-sig")
    done_ids = set(existing_df["ID"].astype(str))
    print(f"Resume detected: {len(done_ids)} rows already available.")
else:
    done_ids = set()

work_df = input_df[~input_df["ID"].astype(str).isin(done_ids)].copy()
work_df.reset_index(drop=True, inplace=True)

print(f"Remaining rows: {len(work_df)} / {len(input_df)}")
print()

## 6. Run retrieval-augmented generation

For each batch:
1. retrieve the most relevant text chunks
2. build a grounded prompt
3. generate answers
4. save results immediately to CSV

In [ ]:
print(f"Starting inference for {len(work_df)} prompts...")
start_time = time.time()

for start in range(0, len(work_df), BATCH_SIZE):
    batch_df = work_df.iloc[start:start + BATCH_SIZE]

    chat_texts = []
    batch_metadata = []

    for _, row in batch_df.iterrows():
        question_id = str(row["ID"])
        question_text = row["Prompt"]

        search_tag = question_id.rsplit("-", 1)[0]
        retrieved_docs = db.similarity_search(question_text, k=TOP_K, filter={"tag": search_tag},)

        context = "nothing_found"
        source = "nothing_found"
        law_ref = "nothing_found"

        system_prompt = (
            "You are a precise Austrian tax law assistant. "
            "Answer in one short sentence. "
            "Do not invent legal references.")

        user_prompt = f"Question: {question_text}\n\nAnswer briefly and factually."

        if retrieved_docs:
            best_doc = retrieved_docs[0]
            context = clean_text(best_doc.page_content)[:1200]
            source = best_doc.metadata.get("source", "nothing_found")
            law_ref = extract_law_reference(context)

            system_prompt = (
                "You are a precise Austrian tax law assistant. "
                "Answer in one short sentence. "
                "Use only the provided context. "
                "If the answer is not contained in the context, respond exactly with: "
                "'No reliable answer found in the context.' "
                "Do not invent information.")

            user_prompt = (
                f"Question: {question_text}\n\n"
                f"Context: {context}\n\n"
                "Answer briefly and factually based only on the context.")

        chat_texts.append(build_chat_text(tokenizer, system_prompt, user_prompt))
        batch_metadata.append(
            {"source": source, "law_ref": law_ref, "context": context})

    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    )
    inputs = inputs.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=4)

    batch_results = []

    for i in range(len(batch_df)):
        row = batch_df.iloc[i]

        input_length = inputs["input_ids"].shape[1]
        generated_tokens = outputs[i][input_length:]

        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        answer = clean_text(answer)
        answer = re.sub(r"(?i)^(answer|assistant):\s*", "", answer)

        if answer == "":
            answer = "No reliable answer found."

        metadata = batch_metadata[i]

        if metadata["source"] != "nothing_found":
            final_answer = f"{answer} | {metadata['source']} | {metadata['law_ref']}"
        else:
            final_answer = answer

        batch_results.append(
            {"ID": row["ID"], "Prompt": row["Prompt"], "Answer": final_answer, "Retrieved_Sources": metadata["source"],
             "Retrieved_Refs": metadata["law_ref"], "Retrieved_Context": metadata["context"]})

    result_df = pd.DataFrame(batch_results)
    file_exists = os.path.exists(OUT_PATH)
    result_df.to_csv(OUT_PATH, mode="a", header=not file_exists, index=False, encoding="utf-8-sig")

    processed = min(start + BATCH_SIZE, len(work_df))
    print(f"Progress: {processed}/{len(work_df)} rows processed.")

print()
print(f"Finished. Results saved to: {OUT_PATH}")
print(f"Total runtime: {(time.time() - start_time):.1f} seconds")